In [1]:
import requests

remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch02-downloaded.pdf"

response = requests.get(remote_pdf_url)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
else:
    print("Failed to download the PDF. Status code:", response.status_code)

In [4]:
import pdfplumber

text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(text[0:20])

Einstein’s Patents a


In [5]:
from utils import chunk_text

chunks = chunk_text(text, 500, 40)
print(len(chunks))
print(chunks[0])

89
Einstein’s Patents and Inventions
Asis Kumar Chaudhuri
Variable Energy Cyclotron Centre
1‐AF Bidhan Nagar, Kolkata‐700 064
Abstract: Times magazine selected Albert Einstein, the German born Jewish Scientist as the person of the 20th
century. Undoubtedly, 20th century was the age of science and Einstein’s contributions in unravelling mysteries
of nature was unparalleled. However, few are aware that Einstein was also a great inventor. He and his
collaborators had patented a wide variety of inventions


In [9]:
%load_ext dotenv
%dotenv

import os
from mistralai.client import Mistral

mistral_api_key = os.getenv("MISTRAL_API_KEY", "")
mistral_client = Mistral(api_key=mistral_api_key)

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [ ]:
def embed(texts):
    response = mistral_client.embeddings.create(
        inputs=texts,
        model="mistral-embed",
    )
    return list(map(lambda n: n.embedding, response.data))

# if embeddings already exist, load from file
import os
import json

if os.path.exists("embeddings.json"):
    with open("embeddings.json", "r") as f:
        embeddings = json.load(f)
else:
    embeddings = embed(chunks)
    with open("embeddings.json", "w") as f:
        json.dump(embeddings, f)


print(embeddings[0][0:3])
print(len(embeddings))
print(len(embeddings[0]))


[-0.02801513671875, 0.033233642578125, 0.0141754150390625]
89
1024


In [16]:
from neo4j import GraphDatabase
driver = GraphDatabase.driver("neo4j://localhost:7687", auth=("neo4j", "newpassword"))

In [17]:
driver.execute_query("""CREATE VECTOR INDEX pdf IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding
OPTIONS {indexConfig: {
  `vector.dimensions`: 1024,
  `vector.similarity_function`: 'cosine'
}}""")

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x00000213C02ED790>, keys=[])

In [18]:
# Add to neo4j
cypher_query = '''
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
'''

driver.execute_query(cypher_query, chunks=chunks, embeddings=embeddings)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x00000213C02FEE10>, keys=[])

In [19]:
records, _, _ = driver.execute_query("MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text")

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

Einstein’s Patents and Inventi
[-0.02801513671875, 0.033233642578125, 0.0141754150390625]


In [22]:
question = "At what time was Einstein really interested in experimental works?"
question_embedding = embed([question])[0]

query = '''
CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

for record in similar_records:
    print(record["text"])
    print(record["score"], record["index"])
    print("======")

in nuances of machines and instruments. However, it must also be
emphasized that his main occupation was theoretical physics. The inventions he worked upon were
his diversions. In his unproductive times he used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three major inventions; (i) refrigeration system with Leo Szilard, (ii)
Sound reproduction system with Rudolf Goldschmidt and (iii) automatic camera
0.9145146012306213 44
CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote 

In [27]:
system_message = "You're en Einstein expert, but can only use the provided documents to respond to the questions."
user_message = f"""
Use the following documents to answer the question that will follow:
{[doc["text"] for doc in similar_records]}

---

The question to answer using information only from the above documents: {question}
"""

print("Question:", question)

response = mistral_client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ],
    response_format={"type": "text"}
)
print(response.choices[0].message.content)

Question: At what time was Einstein really interested in experimental works?
According to the provided documents, Einstein was genuinely interested in experimental works during his time at ETH (Eidgenössische Technische Hochschule). The document states: "In his ETH days Einstein was genuinely interested in experimental works. He wrote to his friend, 'most of the time I worked in the physical laboratory, fascinated by the direct contact with observation.'"


In [28]:
try :
    driver.execute_query(f"CREATE FULLTEXT INDEX ftPdfChunk FOR (c:Chunk) ON EACH [c.text]")
except:
    print("Fulltext Index already exists")

In [29]:
hybrid_query = '''
CALL {
    // vector index
    CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // We use 0 as min
    RETURN n.node AS node, (n.score / max) AS score
    UNION
    // keyword index
    CALL db.index.fulltext.queryNodes('ftPdfChunk', $question, {limit: $k})
    YIELD node, score
    WITH collect({node:node, score:score}) AS nodes, max(score) AS max
    UNWIND nodes AS n
    // We use 0 as min
    RETURN n.node AS node, (n.score / max) AS score
}
// dedup
WITH node, max(score) AS score ORDER BY score DESC LIMIT $k
RETURN node, score
'''
similar_hybrid_records, _, _ = driver.execute_query(hybrid_query, question_embedding=question_embedding, question=question, k=4)

for record in similar_hybrid_records:
    print(record["node"]["text"])
    print(record["score"], record["node"]["index"])
    print("======")

in nuances of machines and instruments. However, it must also be
emphasized that his main occupation was theoretical physics. The inventions he worked upon were
his diversions. In his unproductive times he used to work upon on solving mathematical problems (not
related to his ongoing theoretical investigations) or took upon some practical problem. As shown in
Table. 2, Einstein was involved in three major inventions; (i) refrigeration system with Leo Szilard, (ii)
Sound reproduction system with Rudolf Goldschmidt and (iii) automatic camera
1.0 44
CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, 

In [31]:
user_message = f"""
Use the following documents to answer the question that will follow:
{[doc["node"]["text"] for doc in similar_hybrid_records]}

---

The question to answer using information only from the above documents: {question}
"""

print("Question:", question)

response = mistral_client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ],
    response_format={"type": "text"}
)
print(response.choices[0].message.content)

Question: At what time was Einstein really interested in experimental works?
According to the provided documents, Einstein was genuinely interested in experimental works during his time at ETH (Eidgenössische Technische Hochschule). The document states: "In his ETH days Einstein was genuinely interested in experimental works. He wrote to his friend, 'most of the time I worked in the physical laboratory, fascinated by the direct contact with observation.'"
